In [11]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if modified_dft_d3bj := data.get("modified_dft_d3bj"):
            data_dft_d3bj = modified_dft_d3bj.to_numpy()
        else:
            data_dft_d3bj = data_d3bj
        if modified_dft_d3zero := data.get("modified_dft_d3zero"):
            data_dft_d3zero = modified_dft_d3zero.to_numpy()
        else:
            data_dft_d3zero = data_d3zero

        if modified_ai_d3bj := data.get("modified_ai_d3bj"):
            data_ai_d3bj = modified_ai_d3bj.to_numpy()
        else:
            data_ai_d3bj = data_d3bj
        if modified_ai_d3zero := data.get("modified_ai_d3zero"):
            data_ai_d3zero = modified_ai_d3zero.to_numpy()
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = (
                                data_scf[col[0]] - data_cc[col[0]]
                            )
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

print("Summary")
display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 28.642543112975545 kcal/mol, 116 in 3110260_W4_11
  -1 * W4_11-p4  -1 * 3.9123830266762525  4 * W4_11-p  4 * 8.13873153491295
Top 2 AI: 15.835139658709522 kcal/mol, 122 in 3110260_W4_11
  -1 * W4_11-p2  -1 * 0.4423234111163765  2 * W4_11-p  2 * 8.13873153491295
Top 3 AI: 7.358472908810086 kcal/mol, 13 in 3110260_W4_11
  -1 * W4_11-ph3  -1 * 0.9053690542641561  1 * W4_11-p  1 * 8.13873153491295  3 * W4_11-h  3 * 0.041703476053783106
Top 4 AI: 6.986845495994203 kcal/mol, 57 in 3110260_W4_11
  -1 * W4_11-becl2  -1 * 7.059663417749107  1 * W4_11-be  1 * -0.012091999411495635  2 * W4_11-cl  2 * 0.042454960581380874
Top 5 AI: 4.620493889204226 kcal/mol, 41 in 3110260_W4_11
  -1 * W4_11-sif4  -1 * 3.7717739615472965  1 * W4_11-si  1 * 0.04056976587162353  4 * W4_11-f  4 * -0.22232242338213837
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.610892266675364 kcal/mol
Top 5 DFT: 55.139185982690606 kcal

/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.

Summary


data_path       3110260                                            \
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip   
summary        0.198877      0.247006      0.036954      0.032363   
W4_11           0.16156      0.148744      0.028608      0.021439   
G21EA          0.294814      0.109608      0.014994      0.012762   
G21IP          0.151887      0.102494      0.007099      0.006154   
DIPCS10         0.07944      0.107438      0.006365      0.003622   
PA26           0.183548        0.2982      0.036632      0.038429   
SIE4x4         0.070077      0.093996      0.018423      0.001311   
ALKBDE10       0.187007       0.10564      0.061391      0.061711   
YBDE18         0.186926      0.276239      0.058031      0.041601   
AL2X6          0.345115      0.396501      0.008773      0.002336   
HEAVYSB11      0.231863      0.278481      0.015922      0.009043   
NBPRC          0.174247      0.252724       0.04845      0.032482   
ALK8           0.212319      0.187227      0.040648      0.040223   
RC21           0.196255      0.258984      0.075438      0.082447   
G2RC           0.128206      0.181463      0.020456      0.015617   
BH76RC         0.144839       0.14036      0.060358      0.057338   
FH51           0.185359      0.291878      0.036715      0.029176   
TAUT15          0.26756      0.413544      0.073232      0.069453   
DC13           0.245509      0.397856      0.025266      0.019322   
MB16_43         0.38961      0.489942       0.07826      0.079477   
DARC           0.238854      0.432745      0.024154      0.019359   
RSE43          0.129077      0.193985      0.044826      0.034976   
BSR36          0.291378      0.495777      0.005656      0.001683   
CDIE20         0.193758      0.324292      0.044676      0.044435   
ISO34               NaN           NaN           NaN           NaN   
ISOL24              NaN           NaN           NaN           NaN   
C60ISO              NaN           NaN           NaN           NaN   
PArel          0.354378      0.539445      0.079482       0.06207   
BH76           0.144839       0.14036      0.060358      0.057338   
BHPERI          0.12822      0.197659      0.041323      0.031454   
BHDIV10             NaN           NaN           NaN           NaN   
INV24               NaN           NaN           NaN           NaN   
BHROT27             NaN           NaN           NaN           NaN   
PX13                NaN           NaN           NaN           NaN   
WCPT18              NaN           NaN           NaN           NaN   
RG18                NaN           NaN           NaN           NaN   
ADIM6               NaN           NaN           NaN           NaN   
S22                 NaN           NaN           NaN           NaN   
S66                 NaN           NaN           NaN           NaN   
HEAVY28             NaN           NaN           NaN           NaN   
WATER27             NaN           NaN           NaN           NaN   
CARBHB12            NaN           NaN           NaN           NaN   
PNICO23             NaN           NaN           NaN           NaN   
HAL59               NaN           NaN           NaN           NaN   
AHB21               NaN           NaN           NaN           NaN   
CHB6                NaN           NaN           NaN           NaN   
IL16                NaN           NaN           NaN           NaN   
IDISP               NaN           NaN           NaN           NaN   
ICONF               NaN           NaN           NaN           NaN   
ACONF               NaN           NaN           NaN           NaN   
Amino20x4      0.375241      0.662938      0.032944      0.033728   
PCONF21             NaN           NaN           NaN           NaN   
MCONF               NaN           NaN           NaN           NaN   
SCONF               NaN           NaN           NaN           NaN   
UPU23               NaN           NaN           NaN           NaN   
BUT14DIOL           NaN           NaN           NaN           NaN  

MAE


...

data_path    3110260                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        4.355732   7.377209   3.995796   7.414035   3.818076   7.096457   
sub2       17.660464  14.209048  14.387062  11.172939  15.047083  11.187724   
sub3        4.979405   8.749748   6.835036  10.853316   6.243599  10.207157   
sub4             NaN        NaN        NaN        NaN        NaN        NaN   
sub5       12.732778   5.654023    12.7075   5.738052  12.277054   4.944508   

data_path              1424849                        ...    1477959  \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1        17 / 18   3.239113   7.420922   2.966668  ...   3.046364   
sub2          1 / 9  13.142366   12.53052  11.927684  ...   9.389455   
sub3          1 / 7   4.832263   6.984991   5.854163  ...   6.104995   
sub4         0 / 12  11.556831  11.029773  13.901938  ...  13.234994   
sub5          0 / 9  15.335367  11.571826  14.145431  ...  13.667041   

data_path                          159476                                  \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ  DFT_D3BJ   
sub1        7.102094   18 / 18   3.964682   7.420922   3.868929  7.414578   
sub2       10.005295     7 / 9  12.087202   12.53052   10.04631  9.968145   
sub3        7.521852     7 / 7   5.480279   6.984991    6.43503  8.150912   
sub4       13.798614   11 / 12  11.506946  11.029773  14.836255  14.87552   
sub5        7.523212     8 / 9  20.574666  11.571826  15.230239  8.092645   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        3.421741   7.102094   18 / 18  
sub2       10.398962  10.005295     7 / 9  
sub3        6.162506   7.521852     7 / 7  
sub4       13.767672  13.798614   11 / 12  
sub5       15.647331   7.523212     8 / 9  

[5 rows x 28 columns]

wtmad_2


data_path    3110260                                                      \
Disp type         AI        DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1        6.548376  14.218321  6.340195  14.65431  6.068882  14.212224   
sub2       11.489861   8.822553  6.945341  4.488376  7.997865    5.31153   
sub3        1.778836   6.182447  1.990599  6.728201  1.914592    6.57848   
sub4             0.0        0.0       0.0       0.0       0.0        0.0   
sub5        1.912903    0.84943  1.909106  0.862054  1.844438   0.742836   

data_path             1424849                      ...   1477959             \
Disp type Processed        AI       DFT   AI_D3BJ  ... AI_D3ZERO DFT_D3ZERO   
sub1        17 / 18  1.438234  4.040696  1.379994  ...   1.68376   3.989769   
sub2          1 / 9  3.339073  3.635331  2.970755  ...  2.765998   2.545688   
sub3          1 / 7  1.303233  2.611064   1.66299  ...  1.817199   2.873647   
sub4         0 / 12  4.683102  4.159493  6.811273  ...  6.318974    6.62018   
sub5          0 / 9  6.331811   4.66236   6.30221  ...   5.89853    3.19725   

data_path              159476                                          \
Disp type Processed        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO   
sub1        18 / 18  1.805675  4.040696  1.824002  4.111446  1.628396   
sub2          7 / 9  4.219056  3.635331  3.054971  2.376023  3.302036   
sub3          7 / 7  1.502142  2.611064  1.888567  3.040013  1.770581   
sub4        11 / 12  4.379061  4.159493  6.615548  6.856534   6.37219   
sub5          8 / 9  7.607086   4.66236  6.051386  3.511844  6.087667   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        3.989769   18 / 18  
sub2        2.545688     7 / 9  
sub3        2.873647     7 / 7  
sub4         6.62018   11 / 12  
sub5         3.19725     8 / 9  

[5 rows x 28 columns]

Summary of Subset
MAE


data_path    3110260                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       1.283783  29.506863   1.829601  31.211753   1.170543  29.861055   
G21EA       1.308792   9.754522   1.314104   9.751745   1.300673   9.757546   
G21IP       1.073901   8.953334   1.080964   8.946699   1.075434     8.9519   
DIPCS10     1.941904  12.310852   1.932049  12.334099    1.98976  12.264655   
PA26        2.711683   2.200766   2.800009   1.900152   2.789493    2.00824   
SIE4x4       2.85344  21.908516   3.224883  22.337814   3.282441  22.339928   
ALKBDE10    0.593435  18.125246     0.5386  18.838948   0.583093  18.135588   
YBDE18      5.074164   8.145393   3.228603   7.265227   3.637949   7.472761   
AL2X6       8.936535   5.659145   3.155039   1.332003   5.107285   1.829895   
HEAVYSB11   2.252862   5.391476   2.270553   8.010474   0.894766   5.984154   
NBPRC       4.351381    2.23255     5.1807   2.544088   4.534255   2.274234   
ALK8        4.591736   4.400063   3.698529   3.174803   2.668097   2.559066   
RC21        2.930378   4.820926   2.583504   6.671006   2.257191     6.0297   
G2RC        2.352266   5.917203   2.761017   6.933515   2.539796   6.417298   
BH76RC      0.955492   3.484561   0.819127   3.539828   0.924793   3.536982   
FH51        2.567083   2.916822   2.413515   3.385014   2.209178   3.155435   
TAUT15      2.714948    2.16453    2.74366   2.175485   2.688323   2.145937   
DC13       11.785004  13.090861  12.535288  12.474047  11.200103  11.475687   
MB16_43    16.432498  15.852413  27.007341  33.975204  16.071406  21.536477   
DARC        12.38055  10.505453    4.86861   3.347456    7.34017   5.465073   
RSE43        2.37684   3.042021   2.186353    2.83457    2.05159   2.700194   
BSR36      12.976767   8.400969   5.623455   1.213694   7.654904   3.079106   
CDIE20      1.823816   1.695626   1.572399   1.413647   1.601785   1.402871   
ISO34              0          0          0          0          0          0   
ISOL24             0          0          0          0          0          0   
C60ISO             0          0          0          0          0          0   
PArel       3.695566   1.738616   3.554205   1.659679   3.714653   1.539787   
BH76        2.578501   9.107946   2.863882     9.8892   2.760177   9.676053   
BHPERI      7.380308   8.391549  10.806191  11.817432    9.72702  10.738261   
BHDIV10            0          0          0          0          0          0   
INV24              0          0          0          0          0          0   
BHROT27            0          0          0          0          0          0   
PX13               0          0          0          0          0          0   
WCPT18             0          0          0          0          0          0   
RG18               0          0          0          0          0          0   
ADIM6              0          0          0          0          0          0   
S22                0          0          0          0          0          0   
S66                0          0          0          0          0          0   
HEAVY28            0          0          0          0          0          0   
WATER27            0          0          0          0          0          0   
CARBHB12           0          0          0          0          0          0   
PNICO23            0          0          0          0          0          0   
HAL59              0          0          0          0          0          0   
AHB21              0          0          0          0          0          0   
CHB6               0          0          0          0          0          0   
IL16               0          0          0          0          0          0   
IDISP              0          0          0          0          0          0   
ICONF              0          0          0          0          0          0   
ACONF              0          0          0        

In [2]:
# import numpy as np
np.max(mean_absolute_deviation_list), np.sum(mean_absolute_deviation_list), np.mean(
    mean_absolute_deviation_list
)

(np.float64(1207.482739432191),
 np.float64(98414.09024557225),
 np.float64(71.1084467092285))

In [5]:
mole_name

'BH76RC-NH'

In [9]:
e1 = -4.6025270679642568e02  # orca fc DLPNO-CCSD
e2 = -4.6025221729046751e02  # orca fc CCSD
e3 = -4.6025767731174767e02 # orca nfc CCSD
e4 = -4.6025796979352134e02 # orca nfc DLPNO-CCSD
e = -460.257677434825  # pyscf CCSD
print((np.array([e1, e2, e3, e4]) - e) * 627.5094733748099)
print((np.array([e1, e2, e3, e4]) - e) / e)

[ 3.11912268e+00  3.42629231e+00  7.72321803e-05 -1.83457852e-01]
[-1.07996860e-05 -1.18632336e-05 -2.67409583e-10  6.35206561e-07]


In [14]:
e = -460.2602125077868 # pyscf CCSD(t)
e1 = -4.6026021233042377e02 # orca nfc CCSD(t)
e2 = -4.6026021233042360e02 # orca old nfc CCSD(t)
e3 = -4.6026042971888393e02  # orca nfc DLPNO-CCSD(t)
e4 = -4.6025500524617820e02
print((np.array([e1, e2, e3, e4]) - e) * 627.5094733748099)
print((np.array([e1, e2, e3, e4]) - e) / e)

[ 1.11296967e-04  1.11297074e-04 -1.36302021e-01  3.26760599e+00]
[-3.85353766e-10 -3.85354136e-10  4.71931076e-07 -1.13137340e-05]


In [ ]:
np.array([[-4.366423914512e-02], [0.000000000000e00], [-5.569827800646e-01]])
np.array([[-4.366423911029e-02], [0.000000000000e00], [-5.569827799431e-01]])

[[-4.366423908914e-02], [0.000000000000e00], [-5.569827798789e-01]]

array([[-3.48300000e-11],
       [ 0.00000000e+00],
       [-1.21500032e-10]])